# NND Downstream Visualization & Permutation Analysis

## Purpose
Executes exploratory visualization and non-parametric permutation testing across the harmonized clinical-genomic dataset. This component handles demographic overview mapping, pairwise disease cohort risk transformations, mutation-stratified deep dives, and complex cross-disorder genetic contrast matrices.

## Input
* `output/merged_data.csv`: Consolidated master clinical-genetic dataset (`merged_df`) providing structural core variables.
* `CASE_DATA_PATH` (`/home/../PRS/gwas_cases.txt`): Reference summary file containing empirical GWAS sample sizes for background chart overlays.

## Process
1. **Demographic Grid Mapping:** Renders parallel horizontal violin spreads showing continuous age-at-death tracking across neuropathological categories alongside discrete sample-count marginal charts.
2. **Pairwise Resampling Testing:** Employs non-parametric label-shuffling algorithms ($n = 10,000$) to evaluate mean/median continuous metric drift across custom target vectors (e.g., control vs. single disease cohorts; intra-mutation clusters).
3. **Heatmap & Contrast Matrix Scaling:** Projects continuous case-vs-control Z-score deviations via specialized mapping frameworks (`scattermap`), applying multi-test corrections (Benjamini–Hochberg FDR) to dynamically calculate marker size scales and asterisk overlays.
4. **Cohort Stratification Routing:** Splits underlying data matrices sequentially by specific diagnostic domains (FTLD sub-phenotypes) and biological sex to construct downstream target graphs.

## Output Visualizations
* `output/fig1_overview.pdf`: Global overview chart tracking age distributions split by biological sex next to sample observation counts.
* `output/fig2_prs_violin_disorders_horizontal_[TRAIT].pdf`: Horizontal horizontal box-violin composites featuring automated pairwise significance stars.
* `output/prs_sbayesrc_disorders.pdf`: Main cross-disorder dot-matrix heatmap displaying effect sizes, sample volumes, and heritability bars.
* `output/prs_dis_vs_dis_permutation.pdf`: Symmetrical cross-cohort median matrix heatmap testing targeted primary traits against remaining clinical groups.
* `output/prs_sbayesrc_ftd_subtypes_dotplot.pdf`: Deep-dive statistical dot-matrix plotting frontotemporal lobar degeneration variants.
* `output/prs_sbayesrc_disorders_sex_[M/F].pdf`: Stratified multi-trait dotmaps detailing male- and female-specific genomic trend lines.

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import itertools
# import matplotlib.colors as mcolors
# import matplotlib as mpl
# from statsmodels.stats.multitest import multipletests
# from scattermap import scattermap

# # ==============================================================================
# # STAGE 0: GLOBAL PARAMETERS & PRE-REQUISITE DEFINITIONS (DUMMY WORKSPACE)
# # ==============================================================================

# # --- Upstream Clinical & Genetic Datasets ---
# if 'merged_df' not in globals():
#     # Primary merged DataFrame containing genomic scores (PRS) and clinical designations
#     merged_df = pd.DataFrame({
#         'donor_id': [f'NND-{i}' for i in range(1, 101)],
#         'nd_simple': np.random.choice(["CON", "AD", "PD", "FTD", "MS", "MND"], size=100),
#         'AD_APOE': np.random.normal(loc=0.0, scale=1.0, size=100),
#         'FTD': np.random.normal(loc=0.0, scale=1.0, size=100),
#         'MS': np.random.normal(loc=0.0, scale=1.0, size=100)
#     })

# if 'table1_interest' not in globals():
#     # Diagnostic cohorts of interest for downstream analysis
#     table1_interest = ["CON", "AD", "PD", "FTD", "MS"]

# # --- Plot Matrix Variables (Required for the Scattermap visualization) ---
# if 'median_df_con_difference' not in globals():
#     # Matrix representing the difference in median PRS scores between cases and controls
#     median_df_con_difference = pd.DataFrame(
#         np.random.uniform(-0.4, 0.4, size=(5, 5)),
#         index=["CON", "AD", "PD", "FTD", "MS"],
#         columns=["AD_APOE", "FTD", "MS", "PD", "ALS"]
#     )

# if 'dot_size_df_log' not in globals():
#     # Matrix mapping significance values to dot sizes (e.g. -log10 of FDR corrected p-values)
#     dot_size_df_log = pd.DataFrame(
#         np.random.choice([100, 300, 500, 800], size=(5, 5)),
#         index=["CON", "AD", "PD", "FTD", "MS"],
#         columns=["AD_APOE", "FTD", "MS", "PD", "ALS"]
#     )

# if 'binary_stats_df' not in globals():
#     # Binary threshold matrix mapping which elements receive significance stars (1=Yes, 0=No)
#     binary_stats_df = pd.DataFrame(
#         np.random.choice([0, 1], p=[0.8, 0.2], size=(5, 5)),
#         index=["CON", "AD", "PD", "FTD", "MS"],
#         columns=["AD_APOE", "FTD", "MS", "PD", "ALS"]
#     )

# if 'star_size' not in globals():
#     # Base size of the significance stars plotted on the map
#     star_size = 250

# # --- Bar Chart Margin Data (For marginal distribution visualization) ---
# if 'diagnosis_bar_df' not in globals():
#     # Reference metrics summarizing demographics or observations per diagnostic category
#     diagnosis_bar_df = pd.DataFrame({
#         'Number_of_Observations': [20, 50, 45, 30, 25]
#     }, index=["CON", "AD", "PD", "FTD", "MS"])

# if 'diagnosis_bar' not in globals():
#     # Array of demographic counts/values mapped to the primary y-axis
#     diagnosis_bar = [10, 12, 15, 8, 9]

# if 'diag_pos' not in globals():
#     # Spatial positioning coordinates for alignment of diagnosis bars
#     diag_pos = [0, 1, 2, 3, 4]

# if 'positions' not in globals():
#     # Coordinate layout mapping the horizontal subplot components
#     positions = list(range(12))

# if 'heritability_bar' not in globals():
#     # Array mapping heritability parameters or vertical metrics corresponding to each column
#     heritability_bar = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01, 0.0, 0.0, 0.0, 0.0]

# if 'heritability_bar_df' not in globals():
#     # Meta DataFrame containing the heritability properties mapped on x-axis projections
#     heritability_bar_df = pd.DataFrame({'value': [0.1, 0.2, 0.3, 0.4, 0.5]})

# # --- Label Arrays & Scale Limits ---
# if 'prs_list' not in globals():
#     # List representing overall PRS traits included on the x-axis
#     prs_list = ["AD_APOE", "FTD", "MS", "PD", "ALS"]

# if 'plot_diag_order' not in globals():
#     # Categorical sorted indexing used to order rows on the y-axis
#     plot_diag_order = ["CON", "AD", "PD", "FTD", "MS"]

# if 'min_dot_size' not in globals():
#     # Lower graphical mapping boundary of dot sizes
#     min_dot_size = 100

# if 'max_dot_size' not in globals():
#     # Upper graphical mapping boundary of dot sizes
#     max_dot_size = 800

# if 'min_val' not in globals():
#     # Legend label value mapping the lower significance range limit
#     min_val = 0.05

# if 'max_val' not in globals():
#     # Legend label value mapping the upper significance range limit
#     max_val = 0.0001


# Analysis: Overview Violin Plot (Figure 1.b) <a name="3-analysis-overview-violin-plot-figure-1b"></a>


In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Patch
from matplotlib import gridspec
import matplotlib as mpl

In [ ]:
# --- File Paths ---
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = "fig1_overview.pdf"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# --- Required Categories & Order (y-axis) ---
overview_order = ['CON','AD','PD','DLB', 'VD',
                  'FTD', 'FTD-TDP + MND','MND', 'PSP', 'CBD','AGD', # FTLD
                  'MS', 'MSA', "ATAXIA",
                  'MDD','BP','Other PSYCH',#SCZ # PSYCH
                 'AD + DLB','SICC', 'DLB + SICC', 'AD + ILBD','ILBD', 
         'VE','AD + VE', 'AD + PD',  'AD + CA' ,"HD","Other dementia", "HIP",
                  "Vascular pathology","TAU","Brain cancer","Other Neuropath",# neuropathology
                  "Other (excluded)"
       ]

# --- Category Groupings (Sets) ---
ftld_set = {'FTD', 'FTD-TDP + MND', 'MND', 'PSP', 'CBD','AGD'}
psych_set = {'MDD', 'BP',  'Other PSYCH'}#'SCZ',
dems_set = {'AD + DLB','SICC', 'DLB + SICC', 'AD + ILBD','ILBD', 
         'VE','AD + VE', 'AD + PD',"Other dementia","HD","HIP" ,'AD + CA'}
np_set = {"Vascular pathology","TAU","Brain cancer","Other Neuropath"}
other_set = {"Other (excluded)"}

# --- Input Data Validation Placeholder ---
# Uncomment if you have saved the data in previous step
# Load the dataframe
merged_df = pd.read_csv("output/merged_data.csv")
print("Data loaded successfully.")



In [ ]:




# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def add_diagonal_box(ax, x, y, size_x=1.2, size_y=0.6,
                     color1="coral", color2="white",
                     edgecolor="black", zorder=6):
    """Draws a two-color diagonal rectangle at specific data coordinates."""
    half_x, half_y = size_x / 2.0, size_y / 2.0
    left, right = x - half_x, x + half_x
    bottom, top = y - half_y, y + half_y

    # Upper-left triangle
    tri1 = Polygon([[left, top], [left, bottom], [right, top]],
                   facecolor=color1, edgecolor=edgecolor, zorder=zorder, transform=ax.transData)
    # Bottom-right triangle
    tri2 = Polygon([[right, top], [left, bottom], [right, bottom]],
                   facecolor=color2, edgecolor=edgecolor, zorder=zorder, transform=ax.transData)
    ax.add_patch(tri1)
    ax.add_patch(tri2)


def assign_group(x):
    """Maps diagnostic categories to high-level analysis groups."""
    if x in ftld_set: return "FTLD"
    if x in psych_set: return "PSYCH"
    if x in dems_set: return "Mixed and rare dementias"
    if x in np_set: return "Neuropathology"
    if x in other_set: return "OTHER"
    return "MAIN"


In [ ]:

# ==============================================================================
# 3. PROCESS 1: Data Preparation
# ==============================================================================

# Filter dataframe and assign group classifications
overview_fig = merged_df[merged_df['nd_simple'].isin(overview_order)].copy()
overview_fig["group"] = overview_fig["nd_simple"].apply(assign_group)

# ==============================================================================
# 4. PROCESS 2: Visualizations
# ==============================================================================

# --- Figure Setup ---
sns.set_theme(style="whitegrid", font_scale=1.4)
fig = plt.figure(figsize=(10, 16))
gs = gridspec.GridSpec(1, 2, width_ratios=[4, 1], wspace=0.05)

ax = fig.add_subplot(gs[0])     # Left pane: Violin Plot
ax_bar = fig.add_subplot(gs[1]) # Right pane: Sample Counts

# --- Draw Violin Plot ---
sex_palette = {"F": "#1f78b4", "M": "#b2df8a"}
sns.violinplot(
    data=overview_fig, y="nd_simple", x="age", hue="most_likely_sex",
    split=True, inner="quartile", linewidth=1, cut=0,
    order=overview_order, orient="h", palette=sex_palette, ax=ax
)

# --- Draw Custom Overlays (Diagonal Boxes & Square Markers) ---
color_map = {
    "FTLD": "coral", "PSYCH": "lightgreen", "Mixed and rare dementias": "navy",
    "Neuropathology": "pink", "OTHER": "grey", "MAIN": "white"
}

# Shift overlay coordinate slightly left of the minimum age
xpos = (overview_fig["age"].min() if not overview_fig.empty else 40) - 5
size_x, size_y = 5, 0.6

for i, cat in enumerate(overview_order):
    group = assign_group(cat)
    if cat in {"FTD", "MND", "PSP"}:
        add_diagonal_box(ax, xpos, i, size_x, size_y, color1="coral", color2="white")
    elif cat in {"MDD", "BP", "Other PSYCH"}:
        add_diagonal_box(ax, xpos, i, size_x, size_y, color1="lightgreen", color2="white")
    elif cat in {"AD + DLB", "PDD"}:
        add_diagonal_box(ax, xpos, i, size_x, size_y, color1="navy", color2="white")
    else:
        ax.scatter([xpos], [i], s=300, marker="s", color=color_map[group], edgecolor="black", zorder=5)

# --- Draw Sample Counts Bar Plot ---
counts = overview_fig['nd_simple'].value_counts()
bar_counts = [counts.get(cat, 0) for cat in overview_order]
ax_bar.barh(overview_order, bar_counts, color="grey")

# Annotate count numbers inside the right pane
for y, value in enumerate(bar_counts):
    ax_bar.text(-120, y, str(value), va='center', ha='left', fontsize=12)

ax_bar.set_yticks([])
ax_bar.set_xticks([])
sns.despine(ax=ax_bar, left=True, bottom=True)
ax_bar.set_ylim(ax.get_ylim())

# ==============================================================================
# 5. OUTPUT: Formatting, Styling, & Saving
# ==============================================================================

ax.legend_.remove()

# Custom Legends
patches = [
    Patch(facecolor=c, edgecolor="black", label=l) for c, l in 
    [("white", "Main disorders"), ("grey", "Other"), ("pink", "Neuropathology"), 
     ("coral", "FTLD-spectrum"), ("navy", "Mixed and rare Dems"), ("lightgreen", "Psychiatric")]
]

fig.legend(handles=ax.get_legend_handles_labels()[0], title="Sex", loc="lower center", bbox_to_anchor=(0.25, 0.01))
fig.legend(handles=patches, title="Analysis groups", loc="lower center", bbox_to_anchor=(0.75, -0.05))

fig.suptitle("Brain disorder distribution in the NND", fontsize=20)
ax.set(ylabel="Neuropathological diagnosis (simplified)", xlabel="Age")
sns.despine(ax=ax)
plt.tight_layout(rect=[0, 0.1, 1, 1])

# Save output (auto-creates folder if missing)
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

mpl.rcParams['pdf.fonttype'] = 42
fig.savefig(OUTPUT_PATH, format="pdf", bbox_inches='tight')
plt.show()



# Analysis: Violin Plot of PRS (Figure 2 a,b,c) <a name="4-analysis-violin-plot-of-prs-figure-2-abc"></a>


In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from statannotations.Annotator import Annotator
from statsmodels.stats.multitest import multipletests

In [ ]:


# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Analysis Setup ---
# Choose your PRS target column of interest here (e.g., "AD_APOE", "FTD", "MS")
SELECTED_PRS = "AD_APOE" 

# Number of iterations for the pairwise permutation testing
N_PERMUTATIONS = 10000

# Random seed for reproducibility
RANDOM_SEED = 42

# --- Grouping and Diagnosis Columns ---
DIAGNOSIS_COL = 'nd_simple'
TARGET_GROUPS = ["CON", "AD", "PD", "FTD", "MS"]

# --- Visualization Palette & Style ---
# Note: Ensure this map contains values for each category in TARGET_GROUPS
COLOR_MAP_VIOLIN = {
    "CON": "#a6cee3", 
    "AD": "#1f78b4", 
    "PD": "#b2df8a", 
    "FTD": "#33a02c", 
    "MS": "#fb9a99"
}

# --- Output File Config ---
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = f"fig2_prs_violin_disorders_horizontal_{SELECTED_PRS}.pdf"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)


# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: A pandas DataFrame containing at least:
   - 'donor_id': Identifier for the donor.
   - 'nd_simple' (or set DIAGNOSIS_COL): Diagnostic categorization.
   - The SELECTED_PRS variable (e.g., 'AD_APOE', 'FTD', 'MS'): Numerical Polygenic Risk Scores.
2. `table1_interest`: A sequence/list of diagnostics utilized for raw data filtering.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")
#
# if 'table1_interest' not in locals():
#     table1_interest = ["CON", "AD", 'PD',"DLB","VD","FTD","MND","ATAXIA",#"AD + DLB",
#     "PSP", "MS", "MSA", "MDD", "BP", "Other PSYCH"]
#     print("Fallback table1_interest list defined.")


In [ ]:


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_test_prs_pairwise(
    df,
    prs_col,
    group1,
    group2,
    diag_col,
    n_permutations=10000,
    seed=42,
    plot=False
):
    """
    Performs a two-sided permutation test to evaluate if the mean difference 
    in Polygenic Risk Scores (PRS) between two groups is statistically significant.
    """
    # Extract clean target metrics per group
    g1 = df.loc[df[diag_col] == group1, prs_col].dropna().values
    g2 = df.loc[df[diag_col] == group2, prs_col].dropna().values

    if len(g1) == 0 or len(g2) == 0:
        raise ValueError(f"No data available for {group1} vs {group2}")

    # Calculate original observed statistic
    observed = g1.mean() - g2.mean()

    # Combine array elements and configure generator for permutation
    all_prs = np.concatenate([g1, g2])
    n_g1 = len(g1)
    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    # Perform permutations
    for i in range(n_permutations):
        perm = rng.permutation(all_prs)
        perm_g1 = perm[:n_g1]
        perm_g2 = perm[n_g1:]
        perm_diffs[i] = perm_g1.mean() - perm_g2.mean()

    if plot:
        plt.figure()
        plt.hist(perm_diffs, bins=30, density=True)
        plt.axvline(observed, linestyle="--", linewidth=2, color='red')
        plt.xlabel("Mean PRS difference")
        plt.ylabel("Density")
        plt.title(f"{prs_col}: {group1} vs {group2}")
        plt.show()

    # Compute two-sided p-value
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed))

    return {
        "observed": observed,
        "perm_diffs": perm_diffs,
        "p_value": p_value
    }


In [ ]:


# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Data Preparation & Filtering ---
# Filter global dataset using the initial cohort selection filter
filtered_scores_main_dagnosis = merged_df[merged_df[DIAGNOSIS_COL].isin(table1_interest)].copy()

# Slice to isolate targets needed for the specific PRS violin plotting sequence
viz_df = filtered_scores_main_dagnosis[['donor_id', DIAGNOSIS_COL, SELECTED_PRS]].copy()
viz_df = viz_df[viz_df[DIAGNOSIS_COL].isin(TARGET_GROUPS)]


# --- Process 2: Pairwise Permutation Testing ---
pairs = list(itertools.combinations(TARGET_GROUPS, 2))
results = {}

for g1, g2 in pairs:
    try:
        res = permutation_test_prs_pairwise(
            df=viz_df,
            prs_col=SELECTED_PRS,
            group1=g1,
            group2=g2,
            diag_col=DIAGNOSIS_COL,
            n_permutations=N_PERMUTATIONS,
            seed=RANDOM_SEED,
            plot=False
        )
        results[(g1, g2)] = res
    except ValueError:
        continue

# Convert analytical results dictionary to an organized summary DataFrame
results_df = pd.DataFrame([
    {
        "group1": g1,
        "group2": g2,
        "observed_diff": res["observed"],
        "p_value": res["p_value"]
    }
    for (g1, g2), res in results.items()
])

# Compute adjustments for multiple testing errors
results_df["p_fdr"] = multipletests(results_df["p_value"], method="fdr_bh")[1]
results_df["p_bonferroni"] = multipletests(results_df["p_value"], method="bonferroni")[1]

print("--- Pairwise Statistical Evaluation Summary ---")
display(results_df.head(10))


# ==============================================================================
# 5. OUTPUT: Visualizations & Saving File
# ==============================================================================

plt.figure(figsize=(9, 5))

# Generate base Horizontal Violin Plot
ax = sns.violinplot(
    data=viz_df,
    y=DIAGNOSIS_COL,
    x=SELECTED_PRS,
    order=TARGET_GROUPS,
    orient='h',
    medianprops={'zorder': 3, 'color': 'black', 'linewidth': 4},
    palette=COLOR_MAP_VIOLIN
)

# Instantiate and configure visual significance annotations
annotator = Annotator(
    ax,
    pairs=pairs,
    data=viz_df,
    y=DIAGNOSIS_COL,
    x=SELECTED_PRS,
    order=TARGET_GROUPS,
    orient='h'
)

annotator.configure(
    test=None,
    text_format='star',
    loc='outside',
    hide_non_significant=True,
    pvalue_thresholds=[
        [0.00005, "****"],
        [0.0005, "***"],
        [0.005, "**"],
        [0.05, "*"]
    ],
    verbose=0
)

# Render annotations using the calculated FDR corrected p-values
annotator.set_pvalues_and_annotate(results_df["p_fdr"])

# Reference Lines & Highlight Spans
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)

if "CON" in TARGET_GROUPS:
    con_index = TARGET_GROUPS.index("CON")
    ax.axhspan(
        con_index - 0.5,
        con_index + 0.5,
        color="lightgrey",
        alpha=0.3,
        zorder=0
    )

# Formatting axes labels
plt.ylabel("")
plt.xlabel(f"{SELECTED_PRS} PRS", fontsize=16)
plt.tight_layout()

# PDF Vector/Editable Text Configuration settings
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# Ensure the output target directory directory is generated
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Save visualization figure to disk
plt.savefig(
    OUTPUT_PATH,
    format="pdf",
    bbox_inches='tight'
)

plt.show()


# Analysis: Dot Plot Association of PRS and Diagnosis (Figure 2.d) <a name="5-analysis-dot-plot-association-of-prs-and-diagnosis-figure-2d"></a>


In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import seaborn as sns
from statsmodels.stats.multitest import multipletests

In [ ]:


# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Selected Features and Controls ---
PRS_INTEREST = [
    'AD_APOE', 'PD', 'DLB', 'FTD', 'Neurodegenerative', 
    'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ'
]
CONTROL_LABEL = "CON"
DIAGNOSIS_COL = 'nd_simple'

# --- Permutation Settings ---
N_PERMUTATIONS = 10000
RANDOM_SEED = 42

# --- Visualization Constants ---
MIN_DOT_SIZE = 100
MAX_DOT_SIZE = 300
EPSILON_VAL = 1e-300  # Floor limit to prevent division by zero or infinite log transformations
STAR_COLOR = "#E59400"

# --- File Paths ---
CASE_DATA_PATH = "/home/.../gwas_cases.txt"
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = "prs_sbayesrc_disorders.pdf"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: The comprehensive source DataFrame containing genetic and diagnostic categories.
2. `table1_interest` (mapped as `diagnosis_selector`): Active selection criteria for filtering diagnosis groups.
3. `scattermap`: Ensure you have the custom library or function 'scattermap' loaded in your active environment.
4. `gwas_cases.txt` located at CASE_DATA_PATH: A CSV containing at least 'Disorder' and 'Cases' columns.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")
#
# if 'table1_interest' not in locals():
#     table1_interest = ["CON", "AD", 'PD',"DLB","VD","FTD","MND","ATAXIA",#"AD + DLB",
#     "PSP", "MS", "MSA", "MDD", "BP", "Other PSYCH"]
#     print("Fallback table1_interest list defined.")


In [ ]:

# Connect workspace variables to predefined template variables
diagnosis_selector = table1_interest.copy()
permute_df = merged_df[merged_df[DIAGNOSIS_COL].isin(diagnosis_selector)].copy()


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_test_prs(
    df,
    prs_col,
    group_label,
    diag_col,
    control_label="CON",
    n_permutations=1000,
    seed=42,
    plot=True
):
    """
    Evaluates genetic score distributions via dynamic random shuffling.
    Computes a permutation-derived p-value and mean group differences.
    """
    control = df.loc[df[diag_col] == control_label, prs_col].dropna().values
    group = df.loc[df[diag_col] == group_label, prs_col].dropna().values

    if len(control) == 0 or len(group) == 0:
        raise ValueError(f"No data for {group_label} vs {control_label} in {prs_col}")

    # Observed statistic
    observed = group.mean() - control.mean()

    # Permutations setup
    all_prs = np.concatenate([control, group])
    n_control = len(control)

    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    for i in range(n_permutations):
        perm = rng.permutation(all_prs)
        perm_control = perm[:n_control]
        perm_group = perm[n_control:]
        perm_diffs[i] = perm_group.mean() - perm_control.mean()

    if plot:
        plt.figure()
        plt.hist(perm_diffs, bins=30, density=True)
        plt.axvline(observed, linestyle="--", linewidth=2, color='red')
        plt.xlabel("Mean PRS difference (group − control)")
        plt.ylabel("Density")
        plt.title(f"{prs_col} PRS | {group_label} vs {control_label}")
        plt.show()

    return {
        "observed": observed,
        "perm_diffs": perm_diffs,
        "p_value": np.mean(np.abs(perm_diffs) >= np.abs(observed))
    }



In [ ]:

# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Pairwise Loop & Permutations ---
results = {}
for prs in PRS_INTEREST:
    for group in diagnosis_selector:
        try:
            res = permutation_test_prs(
                df=permute_df,
                prs_col=prs,
                group_label=group,
                diag_col=DIAGNOSIS_COL,
                control_label=CONTROL_LABEL,
                n_permutations=N_PERMUTATIONS,
                seed=RANDOM_SEED,
                plot=False
            )
            results[(prs, group)] = res
        except ValueError:
            print(f"Skipping combination: PRS={prs} | Group={group} | Col={DIAGNOSIS_COL}")
            continue

# --- Process 2: FDR Correction and Wide-Format Structuring ---
keys = list(results.keys())
pvals = np.array([results[k]["p_value"] for k in keys])

# Apply Benjamini–Hochberg False Discovery Rate correction
_, pvals_fdr, _, _ = multipletests(pvals, method="fdr_bh")

# Write adjusted p-values back to our mapping dictionary
for k, p_fdr in zip(keys, pvals_fdr):
    results[k]["p_value_fdr"] = p_fdr

# Pivot data into matrix tables
prs_list = sorted({k[0] for k in results.keys()})
diag_list = sorted({k[1] for k in results.keys()})

pval_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
pval_fdr_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
effect_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)

for (prs, diag), res in results.items():
    pval_df.loc[diag, prs] = res["p_value"]
    pval_fdr_df.loc[diag, prs] = res["p_value_fdr"]
    effect_df.loc[diag, prs] = res["observed"]

stats_df_wide = pval_fdr_df.copy().clip(lower=EPSILON_VAL)

# --- Process 3: Scale Transformations for Plotting Elements ---
# -log10 transformation for improved visual differences in small values
df_log_transformed = -np.log10(stats_df_wide)

min_val = stats_df_wide.min().min()
max_val = stats_df_wide.max().max()

max_val_log = df_log_transformed.min().min()
min_val_log = df_log_transformed.max().max()

# Min-Max Normalization over log values
normalized_df_log = (df_log_transformed - min_val_log) / (max_val_log - min_val_log)
dot_size_df_log = MIN_DOT_SIZE + (MAX_DOT_SIZE - MIN_DOT_SIZE) * (1 - normalized_df_log)

# Binary thresholding map for asterisk overlay markers
binary_stats_df = stats_df_wide.applymap(lambda x: 1 if x < 0.05 else 0)
star_size = binary_stats_df * dot_size_df_log * 0.7

# --- Process 4: Contrast against Control Groups ---
median_df = permute_df.groupby(DIAGNOSIS_COL)[prs_list].median()
con_values = median_df.loc[CONTROL_LABEL]
median_df_con_difference = median_df.subtract(con_values, axis=1).drop(CONTROL_LABEL)

# Reorder indices to align dimensions with designated sorting vectors
plot_diag_order = [group for group in diagnosis_selector if group != CONTROL_LABEL]

dot_size_df_log = dot_size_df_log.reindex(plot_diag_order)[PRS_INTEREST]
median_df_con_difference = median_df_con_difference.reindex(plot_diag_order)[PRS_INTEREST]
binary_stats_df = binary_stats_df.reindex(plot_diag_order)[PRS_INTEREST]
star_size = star_size.reindex(plot_diag_order)[PRS_INTEREST]

# --- Process 5: Loading Background GWAS Metrics ---
heritability_df = pd.read_csv(CASE_DATA_PATH)
heritability_bar_df = heritability_df.set_index("Disorder")

if "Neurodegenerative" not in heritability_bar_df.index:
    heritability_bar_df.loc["Neurodegenerative"] = 0

heritability_bar_df = heritability_bar_df.reindex(PRS_INTEREST)
heritability_bar_df['value'] = heritability_bar_df['Cases'] * -0.00001
heritability_bar = heritability_bar_df['value'].tolist()
positions = np.arange(start=.5, stop=.5 * len(heritability_bar) * 2, step=1)

# Generate cohort percentage profiles for horizontal axis
diagnosis_counts = permute_df[DIAGNOSIS_COL].value_counts()
diagnosis_bar_df = pd.DataFrame(diagnosis_counts).reset_index()
diagnosis_bar_df.columns = ['Diagnosis', 'Number_of_Observations']
diagnosis_bar_df.set_index('Diagnosis', inplace=True).drop(index=CONTROL_LABEL, errors='ignore')

total_observations = diagnosis_bar_df['Number_of_Observations'].sum()
diagnosis_bar_df['prop'] = (diagnosis_bar_df['Number_of_Observations'] / total_observations) * -4

diagnosis_bar_df = diagnosis_bar_df.reindex(plot_diag_order)
diagnosis_bar = diagnosis_bar_df['prop'].tolist()
diag_pos = np.arange(start=.5, stop=.5 * (len(diagnosis_bar)) * 2, step=1)



In [ ]:

# ==============================================================================
# 5. OUTPUT: Visualizations & File Saving
# ==============================================================================

sns.set_theme(
    style="darkgrid", 
    font_scale=1.5, 
    rc={'axes.facecolor': '#F0E6EB', "grid.linestyle": "-", "grid.color": '#b0b0b0'}
)

fig = plt.figure(figsize=(20, 8), dpi=600)

# Establish gridspec framework splits
ax1 = plt.subplot2grid((9, 8), (0, 0), colspan=4, rowspan=9) 
ax2 = plt.subplot2grid((9, 8), (0, 4), colspan=1, rowspan=1) 
ax3 = plt.subplot2grid((9, 8), (3, 4), colspan=1, rowspan=1) 

# Render main continuous difference heatmap (Using external library `scattermap`)
ax1 = scattermap(
    median_df_con_difference,
    cmap=sns.color_palette("Spectral_r", as_cmap=True),
    marker_size=dot_size_df_log,
    ax=ax1,
    vmin=-0.5,
    vmax=0.5,
    linecolor='black',
    linewidths=0.2,
    cbar=True,
    cbar_kws={"shrink": .5}
)

# Render categorical overlay stars indicating threshold significance
star_cmap = mcolors.ListedColormap([STAR_COLOR])
ax1 = scattermap(
    binary_stats_df,
    marker='*',
    marker_size=star_size,
    cbar=False,
    ax=ax1,
    linecolor='black',
    linewidths=0.2,
    cmap=star_cmap
)

# Align axis formats & configurations
ax1.xaxis.tick_top() 
ax1.tick_params('x', labelrotation=90)
ax1.tick_params('y', labelrotation=0)

# Position colorbar label metadata
cbar = ax1.collections[0].colorbar
cbar.ax.text(
    4, 1.2, 
    'diff in Zscore norm \nPRS case vs control', 
    ha='center', va='center', fontsize=12, transform=cbar.ax.transAxes
)

# Horizontal and vertical base limits
ax1.barh(diag_pos, diagnosis_bar, 0.6, alpha=1, edgecolor="none") 
ax1.axvline(x=0, color='k')
ax1.axhline(y=0, color='k')

ax1.set_xlim([min(diagnosis_bar) - 0.1, len(prs_list)])
ax1.bar(list(positions)[:11], heritability_bar[:11], 0.6, color='#41b6c4', bottom=0, edgecolor="none")
ax1.set_ylim([len(plot_diag_order), heritability_bar_df['value'].min() - 0.1])

ax1.set_ylabel('')
ax1.set_xlabel('')

# --- Legending Tricks for Dot Sizes ---
x, y = [2, 5], [2, 5]
a2 = [MIN_DOT_SIZE, MAX_DOT_SIZE]
sc = ax2.scatter(x, y, s=a2, alpha=0.5, c='white')
L = ax2.legend(*sc.legend_elements("sizes"), loc='center right', bbox_to_anchor=(1, 0.7), frameon=False)
L.set_title("FDR corrected p-value", prop={'size': 12})
L.get_texts()[1].set_text(round(min_val, 2))
L.get_texts()[0].set_text(round(max_val, 2))
ax2.axis('off')

# --- Legending Tricks for Star Significance ---
x2, y2 = [4], [2]
a3 = [MAX_DOT_SIZE]
ax3.scatter(x2, y2, s=a3, alpha=1, c=STAR_COLOR, marker='*')
ax3.legend(['significance'], loc='center right', bbox_to_anchor=(1, 1), frameon=False)
ax3.set_ylim([0, 1])
ax3.set_xlim([0, 1])
ax3.axis('off')

# Ensure text formats remain vector-editable when saving to vector types
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['font.family'] = 'sans-serif'

# Build target directory path if not established
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Save visualization
plt.savefig(OUTPUT_PATH, bbox_inches='tight')
plt.show()

# Analysis: FTD violin plot (Figure 3 b,c,d,e)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.stats import zscore
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from statannotations.Annotator import Annotator


In [ ]:

# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Target PRS Score ---
# Choose one of: 'AD_APOE', 'FTD', 'PD', 'ALS'
SELECTED_PRS = 'AD_APOE'

# --- Diagnosis Settings ---
DIAGNOSIS_COL = 'nd_ftd'

# Filter list representing specific FTD subtypes of clinical interest
TARGET_DIAGNOSES = [
    'FTD-TDP-A', 'FTD-TDP-B', 'FTD-TDP-C', 'MND', 'PSP', 'AGD', 'CBD',
    'FTD-TDP-B MND', 'FTD-TAU', 'FTD-PID', 'FTD-FUS', 'FTD_other', 'CON'
]

# --- Grouping & Mutation Profiles ---
MUTATIONS_OF_INTEREST = ['C9orf72_True', 'MAPT_True', 'GRN_True']
MUTATION_ORDER = ['control', 'absent', 'unknown'] + MUTATIONS_OF_INTEREST

# --- Permutation Testing Config ---
N_PERMUTATIONS = 10000
RANDOM_SEED = 42

# --- Visual Boundaries ---
X_LIMITS = (-5, 7)
Y_LIMITS = (5.5, -0.5)

# --- File Paths ---
OUTPUT_DIR = "output"
EXCEL_OUTPUT_NAME = f"sup_ftld_stats_{SELECTED_PRS}.xlsx"
PDF_OUTPUT_NAME = f"{SELECTED_PRS}_violin.pdf"

EXCEL_OUTPUT_PATH = os.path.join(OUTPUT_DIR, EXCEL_OUTPUT_NAME)
PDF_OUTPUT_PATH = os.path.join(OUTPUT_DIR, PDF_OUTPUT_NAME)

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: Master cohort DataFrame containing clinical parameters, mutations, and PRS metrics.
2. `ftd_subtypes` (mapped as `diagnosis_selector`): Subgroup array list containing FTD codes of interest.
3. `prs_interest2`: Set of columns containing numerical PRS values to be scaled using Z-scores.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")
#
# if 'ftd_subtypes' not in locals():
#     ftd_subtypes = ['FTD-TDP-A', 'FTD-TDP-B', 'CON']
#     print("Fallback ftd_subtypes defined.")
#
# if 'prs_interest2' not in locals():
#     prs_interest2 = ['AD_APOE', 'FTD', 'PD', 'ALS']



In [ ]:

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_pvalue(x, y, n_permutations=10000, seed=42):
    """
    Computes a two-sided permutation p-value to evaluate the differences 
    between two mean distributions.
    """
    x = np.asarray(x)
    y = np.asarray(y)

    # Calculate observed difference
    observed = y.mean() - x.mean()

    all_vals = np.concatenate([x, y])
    n_x = len(x)

    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    # Shuffling loops
    for i in range(n_permutations):
        perm = rng.permutation(all_vals)
        perm_x = perm[:n_x]
        perm_y = perm[n_x:]
        perm_diffs[i] = perm_y.mean() - perm_x.mean()

    # Calculate two-sided permutation p-value
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed))

    return observed, p_value



In [ ]:

# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Isolation and Standardization ---
# Slice and standardize selected polygenic score columns
filtered_scores_ftd = merged_df[merged_df[DIAGNOSIS_COL].isin(ftd_subtypes)].copy()
filtered_scores_ftd[prs_interest2] = filtered_scores_ftd[prs_interest2].apply(zscore)
permute_df = filtered_scores_ftd

# --- Process 2: Categorization Mapping ---
# Fill genetic mutations mapping
viz_df = permute_df[['donor_id', DIAGNOSIS_COL, 'genetic_mutation', SELECTED_PRS]].copy()
viz_df['genetic_mutation'] = viz_df['genetic_mutation'].fillna('unknown')

# Keep only target cohort diagnoses
viz_df = viz_df[viz_df[DIAGNOSIS_COL].isin(TARGET_DIAGNOSES)]

# Establish grouping categories
viz_df['plot_group'] = viz_df['genetic_mutation']
viz_df.loc[viz_df[DIAGNOSIS_COL] == 'CON', 'plot_group'] = 'control'

# Build final plot dataframe
plot_df = viz_df[viz_df['plot_group'].isin(MUTATION_ORDER)].copy()


# --- Process 3: Pairwise Permutation Testing ---
results = []
for g1, g2 in combinations(MUTATION_ORDER, 2):
    x = plot_df.loc[plot_df['plot_group'] == g1, SELECTED_PRS].dropna()
    y = plot_df.loc[plot_df['plot_group'] == g2, SELECTED_PRS].dropna()
    
    if len(x) > 0 and len(y) > 0:
        diff, p = permutation_pvalue(
            x, y, 
            n_permutations=N_PERMUTATIONS, 
            seed=RANDOM_SEED
        )
        results.append({
            'Group1': g1,
            'Group2': g2,
            'n1': len(x),
            'n2': len(y),
            'mean_diff': diff,
            'p': p
        })

results_df = pd.DataFrame(results)

# --- Process 4: Multiple Testing FDR Corrections ---
results_df['p_FDR_val'] = multipletests(results_df['p'], method='fdr_bh')[1]
results_df['p_FDR'] = results_df['p_FDR_val'].map('{:.3g}'.format)

# Ensure the output directories exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

results_df.to_excel(EXCEL_OUTPUT_PATH, index=False)



In [ ]:

# ==============================================================================
# 5. OUTPUT: Formatting, Styling, & Saving
# ==============================================================================

plt.figure(figsize=(8, 5), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

# Vertical reference line
plt.axvline(x=0, color='red', linestyle='--', linewidth=0.5)

# Render base Violin Plot
sns.violinplot(
    data=plot_df,
    y='plot_group',
    x=SELECTED_PRS,
    order=MUTATION_ORDER,
    inner="box",
    ax=ax
)

# Render Strip Plot overlays
sns.stripplot(
    data=plot_df,
    y='plot_group',
    x=SELECTED_PRS,
    order=MUTATION_ORDER,
    jitter=True,
    size=5,
    color='black',
    alpha=0.7,
    ax=ax
)

# Render Median markers
medians = plot_df.groupby('plot_group')[SELECTED_PRS].median()
for i, group in enumerate(MUTATION_ORDER):
    plt.scatter(
        medians[group], i, 
        color='white', 
        s=80, 
        zorder=5
    )

# --- Process Significance Lines Display ---
sig_results = results_df[results_df['p_FDR_val'] < 0.05]

x_max = plot_df[SELECTED_PRS].max()
x_min = plot_df[SELECTED_PRS].min()
x_range = x_max - x_min

# Define geometric offset values for text/brackets
line_start = x_max + 0.1 * x_range
step = 0.08 * x_range

for i, row in enumerate(sig_results.itertuples()):
    g1 = row.Group1
    g2 = row.Group2
    pval = row.p_FDR

    y1 = MUTATION_ORDER.index(g1)
    y2 = MUTATION_ORDER.index(g2)

    x_coord = line_start + i * step
    
    # Draw bracket line
    plt.plot(
        [x_coord, x_coord + 0.02 * x_range, x_coord + 0.02 * x_range, x_coord],
        [y1, y1, y2, y2],
        lw=1,
        c='black'
    )

    # Overlay FDR value labels
    plt.text(
        x_coord + 0.025 * x_range,
        (y1 + y2) / 2,
        f"FDR={pval}",
        va='center',
        fontsize=8
    )

# Formatting axes spines and ticks
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)
    spine.set_color("black")    

ax.xaxis.set_ticks_position('bottom')
ax.tick_params(
    axis='both',
    which='major',
    direction='out',
    length=6,
    width=1.2,
    colors='black'
)

plt.xlim(X_LIMITS)
plt.ylim(Y_LIMITS)
plt.ylabel(SELECTED_PRS)
plt.xlabel("Mutation group")
plt.title(f"{SELECTED_PRS} PRS by mutation")

# Ensure text formats remain vector-editable when saving to PDF
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(PDF_OUTPUT_PATH, bbox_inches="tight")
plt.show()

# Analysis: FTD dot plot (Figure 3 a)

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import seaborn as sns
from utils import REPLACE_DICT_FTD
from scipy.stats import zscore
from statsmodels.stats.multitest import multipletests

# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Analysis Settings ---
DIAGNOSIS_COL = 'nd_ftd'  # Generated FTD column
N_PERMUTATIONS = 10000
RANDOM_SEED = 42

# --- Subset Selections ---
# Filter list representing specific FTD subtypes of clinical interest
DIAGNOSIS_SELECTOR = [
    'FTD-TDP-A', 'FTD-TDP-B', 'FTD-TDP-C', 'MND', 'PSP', 'AGD', 'CBD',
    'FTD-TDP-B MND', 'FTD-TAU', 'FTD-PID', 'FTD-FUS', 'FTD_other', 'CON'
]

# PRS profiles list of interest (prs_interest2)
PRS_INTEREST = [
    'AD_APOE', 'PD', 'DLB', 'FTD', 'Neurodegenerative', 
    'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ'
]

# --- Visual Styling Configuration ---
MIN_DOT_SIZE = 100
MAX_DOT_SIZE = 300
EPSILON_VAL = 1e-300  # Safe minimum floor for -log10 transformation
STAR_COLOR = "#E59400"

# --- File Paths ---
CASE_DATA_PATH = "/home/../PRS/gwas_cases.txt"
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = "prs_sbayesrc_ftd_subtypes_dotplot.pdf"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: Master pandas DataFrame containing biological sex, `nd_clean` neuropathology, and continuous PRS scores.
2. `scattermap`: Ensure you have the custom 'scattermap' plotting package loaded or defined in your environment.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_test_prs(
    df,
    prs_col,
    group_label,
    diag_col,
    control_label="CON",
    n_permutations=1000,
    seed=42
):
    """
    Evaluates distribution differences between an FTD subtype cohort and control 
    using randomized permutation shuffles.
    """
    control = df.loc[df[diag_col] == control_label, prs_col].dropna().values
    group = df.loc[df[diag_col] == group_label, prs_col].dropna().values

    if len(control) == 0 or len(group) == 0:
        raise ValueError(f"No data for {group_label} vs {control_label} in {prs_col}")

    observed = group.mean() - control.mean()
    all_prs = np.concatenate([control, group])
    n_control = len(control)

    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    for i in range(n_permutations):
        perm = rng.permutation(all_prs)
        perm_control = perm[:n_control]
        perm_group = perm[n_control:]
        perm_diffs[i] = perm_group.mean() - perm_control.mean()

    return {
        "observed": observed,
        "perm_diffs": perm_diffs,
        "p_value": np.mean(np.abs(perm_diffs) >= np.abs(observed))
    }


# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Dynamic Generation of `nd_ftd` & Data Filtering ---
merged_df[DIAGNOSIS_COL] = merged_df['nd_clean'].replace(REPLACE_DICT_FTD)

# Filter cohort to only contain valid selected FTD-relevant diagnoses
permute_df = merged_df[merged_df[DIAGNOSIS_COL].isin(DIAGNOSIS_SELECTOR)].copy()

# Z-score standardize selected polygenic score columns over the clean subset
permute_df[PRS_INTEREST] = permute_df[PRS_INTEREST].apply(zscore)


# --- Process 2: Pairwise Loop Permutation Testing ---
results = {}
for prs in PRS_INTEREST:
    for group in DIAGNOSIS_SELECTOR:
        try:
            res = permutation_test_prs(
                df=permute_df,
                prs_col=prs,
                group_label=group,
                diag_col=DIAGNOSIS_COL,
                control_label="CON",
                n_permutations=N_PERMUTATIONS,
                seed=RANDOM_SEED
            )
            results[(prs, group)] = res
        except ValueError:
            continue

# --- Process 3: Multiple Testing FDR Corrections ---
keys = list(results.keys())
pvals = np.array([results[k]["p_value"] for k in keys])

_, pvals_fdr, _, _ = multipletests(pvals, method="fdr_bh")

for k, p_fdr in zip(keys, pvals_fdr):
    results[k]["p_value_fdr"] = p_fdr

prs_list = sorted({k[0] for k in results.keys()})
diag_list = sorted({k[1] for k in results.keys()})

pval_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
pval_fdr_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
effect_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)

for (prs, diag), res in results.items():
    pval_df.loc[diag, prs] = res["p_value"]
    pval_fdr_df.loc[diag, prs] = res["p_value_fdr"]
    effect_df.loc[diag, prs] = res["observed"]

stats_df_wide = pval_fdr_df.copy().clip(lower=EPSILON_VAL)

# --- Process 4: Scale Transformations for Dot Dimensions ---
df_log_transformed = -np.log10(stats_df_wide)
min_val = stats_df_wide.min().min()
max_val = stats_df_wide.max().max()

max_val_log = df_log_transformed.min().min()
min_val_log = df_log_transformed.max().max()

normalized_df_log = (df_log_transformed - min_val_log) / (max_val_log - min_val_log)
dot_size_df_log = MIN_DOT_SIZE + (MAX_DOT_SIZE - MIN_DOT_SIZE) * (1 - normalized_df_log)

binary_stats_df = stats_df_wide.applymap(lambda x: 1 if x < 0.05 else 0)
star_size = binary_stats_df * dot_size_df_log * 0.7

# --- Process 5: Calculate Contrast differences vs Control ---
median_df = permute_df.groupby(DIAGNOSIS_COL)[prs_list].median()
con_values = median_df.loc['CON']
median_df_con_difference = median_df.subtract(con_values, axis=1).drop('CON')

plot_diag_order = [diag for diag in DIAGNOSIS_SELECTOR if diag != "CON"]

dot_size_df_log = dot_size_df_log.reindex(plot_diag_order)[PRS_INTEREST]
median_df_con_difference = median_df_con_difference.reindex(plot_diag_order)[PRS_INTEREST]
binary_stats_df = binary_stats_df.reindex(plot_diag_order)[PRS_INTEREST]
star_size = star_size.reindex(plot_diag_order)[PRS_INTEREST]

# --- Process 6: Load GWAS cases & Set Subclass Fallbacks ---
try:
    heritability_df = pd.read_csv(CASE_DATA_PATH)
    heritability_bar_df = heritability_df.set_index("Disorder")
except FileNotFoundError:
    print("Warning: GWAS cases file not found. Generating an empty fallback structure.")
    heritability_bar_df = pd.DataFrame(columns=["Cases"])

# Safely align the index to contain each target FTD subclass without NaN values
for prs_name in PRS_INTEREST:
    if prs_name not in heritability_bar_df.index:
        heritability_bar_df.loc[prs_name, "Cases"] = 0

heritability_bar_df = heritability_bar_df.reindex(PRS_INTEREST)
heritability_bar_df['value'] = heritability_bar_df['Cases'] * -0.00001
heritability_bar = heritability_bar_df['value'].tolist()
positions = np.arange(start=.5, stop=.5 * len(heritability_bar) * 2, step=1)

# Generate cohort percentage distribution bars
diagnosis_counts = permute_df[DIAGNOSIS_COL].value_counts()
diagnosis_bar_df = pd.DataFrame(diagnosis_counts).reset_index()
diagnosis_bar_df.columns = ['Diagnosis', 'Number_of_Observations']
diagnosis_bar_df.set_index('Diagnosis', inplace=True).drop(index='CON', errors='ignore')

total_observations = diagnosis_bar_df['Number_of_Observations'].sum()
diagnosis_bar_df['prop'] = (diagnosis_bar_df['Number_of_Observations'] / total_observations) * -4

diagnosis_bar_df = diagnosis_bar_df.reindex(plot_diag_order)
diagnosis_bar = diagnosis_bar_df['prop'].tolist()
diag_pos = np.arange(start=.5, stop=.5 * (len(diagnosis_bar)) * 2, step=1)


# ==============================================================================
# 5. OUTPUT: Visualizations & Saving File
# ==============================================================================

sns.set_theme(
    style="darkgrid", 
    font_scale=1.5, 
    rc={'axes.facecolor': '#F0E6EB', "grid.linestyle": "-", "grid.color": '#b0b0b0'}
)

fig = plt.figure(figsize=(20, 8), dpi=600)

ax1 = plt.subplot2grid((9, 8), (0, 0), colspan=4, rowspan=9) 
ax2 = plt.subplot2grid((9, 8), (0, 4), colspan=1, rowspan=1) 
ax3 = plt.subplot2grid((9, 8), (3, 4), colspan=1, rowspan=1) 

# Scatter heatmap using 'scattermap'
ax1 = scattermap(
    median_df_con_difference,
    cmap=sns.color_palette("Spectral_r", as_cmap=True),
    marker_size=dot_size_df_log,
    ax=ax1,
    vmin=-0.5,
    vmax=0.5,
    linecolor='black',
    linewidths=0.2,
    cbar=True,
    cbar_kws={"shrink": .5}
)

# Overlay categorical asterisks for significance
star_cmap = mcolors.ListedColormap([STAR_COLOR])
ax1 = scattermap(
    binary_stats_df,
    marker='*',
    marker_size=star_size,
    cbar=False,
    ax=ax1,
    linecolor='black',
    linewidths=0.2,
    cmap=star_cmap
)

ax1.xaxis.tick_top() 
ax1.tick_params('x', labelrotation=90)
ax1.tick_params('y', labelrotation=0)

cbar = ax1.collections[0].colorbar
cbar.ax.text(
    4, 1.2, 
    'diff in Zscore norm \nPRS case vs control', 
    ha='center', va='center', fontsize=12, transform=cbar.ax.transAxes
)

# Render side bar charts
ax1.barh(diag_pos, diagnosis_bar, 0.6, alpha=1, edgecolor="none") 
ax1.axvline(x=0, color='k')
ax1.axhline(0, color='k')

ax1.set_xlim([min(diagnosis_bar) - 0.1, len(prs_list)])
ax1.bar(list(positions)[:11], heritability_bar[:11], 0.6, color='#41b6c4', bottom=0, edgecolor="none")
ax1.set_ylim([len(plot_diag_order), heritability_bar_df['value'].min() - 0.1])

ax1.set_ylabel('')
ax1.set_xlabel('')

# --- Legends ---
x, y = [2, 5], [2, 5]
a2 = [MIN_DOT_SIZE, MAX_DOT_SIZE]
sc = ax2.scatter(x, y, s=a2, alpha=0.5, c='white')
L = ax2.legend(*sc.legend_elements("sizes"), loc='center right', bbox_to_anchor=(1, 0.7), frameon=False)
L.set_title("FDR corrected p-value", prop={'size': 12})
L.get_texts()[1].set_text(round(min_val, 2))
L.get_texts()[0].set_text(round(max_val, 2))
ax2.axis('off')

x2, y2 = [4], [2]
a3 = [MAX_DOT_SIZE]
ax3.scatter(x2, y2, s=a3, alpha=1, c=STAR_COLOR, marker='*')
ax3.legend(['significance'], loc='center right', bbox_to_anchor=(1, 1), frameon=False)
ax3.set_ylim([0, 1])
ax3.set_xlim([0, 1])
ax3.axis('off')

# Keep vector elements editable
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['font.family'] = 'sans-serif'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

plt.savefig(OUTPUT_PATH, bbox_inches='tight')
plt.show()


# Analysis: Heatmap Association of PRS and Diagnosis (Figure Extended 2.a) <a name="6-analysis-heatmap-association-of-prs-and-diagnosis-figure-extended-2a"></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from statsmodels.stats import multitest

In [ ]:


# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Analysis Variables ---
DIAGNOSIS_COL = 'nd_simple'
N_PERMUTATIONS = 10000
RANDOM_SEED = 42

# --- PRS Target Map ---
# Links polygenic risk scores to their respective target "primary" clinical disorder
TESTING_DICT = {
    'AD_APOE': 'AD',
    'PD': 'PD',
    'DLB': 'DLB',
    'FTD': 'FTD',
    'ALS': 'MND',
    'MS': 'MS',
    'MSA': 'MSA',
    'MDD': 'MDD',
    'BP': 'BP',
}

# --- Target Order (Columns / Y-Axis Order) ---
TABLE1_INTEREST = [
    "CON", "AD", "PD", "DLB", "VD", "FTD", "MND", "ATAXIA", 
    "PSP", "MS", "MSA", "MDD", "BP", "Other PSYCH"
]

# --- Subset of PRS list ---
PRS_INTEREST = [
    'AD_APOE', 'PD', 'DLB', 'FTD', 'Neurodegenerative', 
    'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ'
]

# --- Output Directory & File Configuration ---
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = "prs_dis_vs_dis_permutation.pdf"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)


# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: Global master dataset containing diagnostic cohorts and individual genetic scores.
   - Alternatively, it can be optionally loaded in the block below.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")

# Initialize local variable pointing to data
filtered_scores_main_dagnosis = merged_df.copy()



In [ ]:

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_test_two_groups(x, y, n_permutations=10000, seed=42):
    """
    Computes a two-tailed permutation p-value comparing median differences
    between two distributions.
    """
    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]

    if len(x) == 0 or len(y) == 0:
        raise ValueError("Cannot perform permutation test with empty groups")

    # Observed statistic: Difference of medians
    observed = np.median(x) - np.median(y)

    combined = np.concatenate([x, y])
    n_x = len(x)

    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    # Permutation sampling loop
    for i in range(n_permutations):
        perm = rng.permutation(combined)
        perm_x = perm[:n_x]
        perm_y = perm[n_x:]
        perm_diffs[i] = np.median(perm_x) - np.median(perm_y)

    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed))

    return p_value


In [ ]:


# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Isolation & Preparation of Target Data ---
# Select only needed columns
dvd = filtered_scores_main_dagnosis[['donor_id', DIAGNOSIS_COL] + PRS_INTEREST].copy()

# Calculate wide median values
median_values = dvd.groupby(DIAGNOSIS_COL)[PRS_INTEREST].median()
median_values = pd.DataFrame(median_values)

# Reindex and clean rows/columns
median_values = median_values.loc[TABLE1_INTEREST, PRS_INTEREST]
median_values = median_values.T
median_values = median_values.drop(index=["Neurodegenerative", "SCZ"], errors="ignore")


# --- Process 2: Running Pairwise Permutations (Target vs Others) ---
results_dict = {}

for prs, disorder in TESTING_DICT.items():
    # Capture PRS values for the primary target disease group
    selected_values = dvd[dvd[DIAGNOSIS_COL] == disorder][prs].values
    other_disorders = dvd[DIAGNOSIS_COL].dropna().unique()

    for other_disorder in other_disorders:
        # Assign a null placeholder value '9' for self-comparison (main diagonal)
        if other_disorder == disorder:
            results_dict[(prs, disorder, other_disorder)] = {
                'pvalue': 9.0,
                'disorder': disorder,
                'other_disorder': other_disorder
            }
        else:
            other_values = dvd[dvd[DIAGNOSIS_COL] == other_disorder][prs].values
            try:
                pvalue = permutation_test_two_groups(
                    selected_values,
                    other_values,
                    n_permutations=N_PERMUTATIONS,
                    seed=RANDOM_SEED
                )
            except ValueError:
                continue

            results_dict[(prs, disorder, other_disorder)] = {
                'pvalue': pvalue,
                'disorder': disorder,
                'other_disorder': other_disorder
            }


# --- Process 3: Metric Transformations & FDR Correction ---
dvd_test_results = pd.DataFrame.from_dict(results_dict, orient='index')

# Filter self-comparison placeholder elements before applying correction
pvals = dvd_test_results['pvalue'].replace(9.0, np.nan)
dvd_test_results['fdr'] = multitest.multipletests(pvals.fillna(1.0), method='fdr_bh')[1]

# Re-apply placeholder indicator back to diagonal elements
dvd_test_results.loc[dvd_test_results['pvalue'] == 9.0, 'fdr'] = 9.0

# Calculate standard Bonferroni correction
num_tests = pvals.notna().sum()
dvd_test_results['bonferroni'] = (dvd_test_results['pvalue'] * num_tests).clip(upper=1.0)
dvd_test_results.index.names = ['prs_index', 'disorder_index', 'other_disorder_index']


# --- Process 4: Pivoting and Index Alignment ---
df_pivoted = dvd_test_results[['fdr']].reset_index()
df_pivoted = df_pivoted.pivot(index='prs_index', columns='other_disorder_index', values='fdr')

# Crucial: Ensure the pivoted p-values match the order of median_values matrix
df_pivoted = df_pivoted.reindex(index=median_values.index, columns=median_values.columns)

# Transpose matrices to match requested visual orientation
df_pivoted = df_pivoted.T
median_values = median_values.T


# --- Process 5: Build Categorical Mask Layers ---
mask_0_000005 = df_pivoted < 0.000005
mask_0_00005 = df_pivoted < 0.00005
mask_0_0005 = df_pivoted < 0.0005
mask_0_005 = df_pivoted < 0.005
mask_0_05 = (df_pivoted >= 0.005) & (df_pivoted < 0.05)
mask_pvalue_9 = df_pivoted == 9.0


# ==============================================================================
# 5. OUTPUT: Visualizations & Saving File
# ==============================================================================

plt.figure(figsize=(8, 6))

# Render basic base Heatmap
ax = sns.heatmap(
    median_values,
    cmap="coolwarm",
    linewidths=0.3,
    linecolor='grey',
    square=False,
    cbar_kws={
        "label": "Median PRS",
        "shrink": 0.8
    }
)

# Render significance annotations sequentially over respective axes nodes
for (i, j), val in np.ndenumerate(df_pivoted.values):
    if mask_0_000005.iloc[i, j]:
        ax.text(j + 0.5, i + 0.5, '*****', ha='center', va='center', fontweight='bold')
    elif mask_0_00005.iloc[i, j]:
        ax.text(j + 0.5, i + 0.5, '****', ha='center', va='center', fontweight='bold')
    elif mask_0_0005.iloc[i, j]:
        ax.text(j + 0.5, i + 0.5, '***', ha='center', va='center', fontweight='bold')
    elif mask_0_005.iloc[i, j]:
        ax.text(j + 0.5, i + 0.5, '**', ha='center', va='center', fontweight='bold')
    elif mask_0_05.iloc[i, j]:
        ax.text(j + 0.5, i + 0.5, '*', ha='center', va='center', fontweight='bold')
    elif mask_pvalue_9.iloc[i, j]:
        ax.text(j + 0.5, i + 0.5, 'o', ha='center', va='center')

# Axes labels formatting
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='center', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

# Clean boundary border frame spines
for spine in ax.spines.values():
    spine.set_visible(False)

plt.title('Heatmap of Median PRS with Permutation Test Significance', fontsize=12, pad=15)
plt.tight_layout()

# PDF Vector font settings
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# Ensure target folder directory is created
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Save visualization
plt.savefig(OUTPUT_PATH, dpi=600, bbox_inches='tight')
plt.show()

# Analysis: Mixed and rare dementia (Figure extended 2 b)

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import seaborn as sns
from statsmodels.stats.multitest import multipletests

In [ ]:


# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Analysis Settings ---
DIAGNOSIS_COL = 'nd_simple'
N_PERMUTATIONS = 10000
RANDOM_SEED = 42

# --- Subset Selections ---
# Diagnosis selector list (e.g. rare_mixed_dems)
DIAGNOSIS_SELECTOR = [
    "CON", "AD + DLB", "SICC", "DLB + SICC", "AD + ILBD", "ILBD", 
    "VE", "AD + VE", "AD + PD", "AD + CA", "HD", "Other dementia", "HIP"
]

# PRS profiles list of interest (prs_interest2)
PRS_INTEREST = [
    'AD_APOE', 'PD', 'DLB', 'FTD', 'Neurodegenerative', 
    'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ'
]

# --- Visual Styling Configuration ---
MIN_DOT_SIZE = 100
MAX_DOT_SIZE = 300
EPSILON_VAL = 1e-300  # Safe minimum floor for -log10 transformation
STAR_COLOR = "#E59400"

# --- File Paths ---
CASE_DATA_PATH = "/home/../PRS/gwas_cases.txt"
OUTPUT_DIR = "output"
OUTPUT_FILE_NAME = "prs_sbayesrc_disorders.pdf"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: Master pandas DataFrame containing biological sex, diagnostics, and continuous PRS scores.
2. `scattermap`: Ensure you have the custom 'scattermap' plotting package loaded or defined in your environment.
3. `gwas_cases.txt` located at CASE_DATA_PATH: A csv file containing 'Disorder' and 'Cases' columns.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")

# Filter initial dataset down to target cohorts
permute_df = merged_df[merged_df[DIAGNOSIS_COL].isin(DIAGNOSIS_SELECTOR)].copy()



In [ ]:

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_test_prs(
    df,
    prs_col,
    group_label,
    diag_col,
    control_label="CON",
    n_permutations=1000,
    seed=42,
    plot=True
):
    """
    Evaluates distribution differences between a case cohort and control cohort 
    using randomized permutation shuffles.
    """
    control = df.loc[df[diag_col] == control_label, prs_col].dropna().values
    group = df.loc[df[diag_col] == group_label, prs_col].dropna().values

    if len(control) == 0 or len(group) == 0:
        raise ValueError(f"No data for {group_label} vs {control_label} in {prs_col}")

    # Calculate original observed statistic
    observed = group.mean() - control.mean()

    # Create shuffled array
    all_prs = np.concatenate([control, group])
    n_control = len(control)

    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    for i in range(n_permutations):
        perm = rng.permutation(all_prs)
        perm_control = perm[:n_control]
        perm_group = perm[n_control:]
        perm_diffs[i] = perm_group.mean() - perm_control.mean()

    if plot:
        plt.figure()
        plt.hist(perm_diffs, bins=30, density=True)
        plt.axvline(observed, linestyle="--", linewidth=2, color='red')
        plt.xlabel("Mean PRS difference (group − control)")
        plt.ylabel("Density")
        plt.title(f"{prs_col} PRS | {group_label} vs {control_label}")
        plt.show()

    return {
        "observed": observed,
        "perm_diffs": perm_diffs,
        "p_value": np.mean(np.abs(perm_diffs) >= np.abs(observed))
    }


In [ ]:


# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Pairwise Loop Permutation Testing ---
results = {}
for prs in PRS_INTEREST:
    for group in DIAGNOSIS_SELECTOR:
        try:
            res = permutation_test_prs(
                df=permute_df,
                prs_col=prs,
                group_label=group,
                diag_col=DIAGNOSIS_COL,
                control_label="CON",
                n_permutations=N_PERMUTATIONS,
                seed=RANDOM_SEED,
                plot=False
            )
            results[(prs, group)] = res
        except ValueError:
            # Skip iterations where categorical group sizes are zero
            continue

# --- Process 2: Multiple Testing FDR Corrections ---
keys = list(results.keys())
pvals = np.array([results[k]["p_value"] for k in keys])

# Apply Benjamini–Hochberg False Discovery Rate correction
_, pvals_fdr, _, _ = multipletests(pvals, method="fdr_bh")

for k, p_fdr in zip(keys, pvals_fdr):
    results[k]["p_value_fdr"] = p_fdr

# Convert summary stats back to wide formats
prs_list = sorted({k[0] for k in results.keys()})
diag_list = sorted({k[1] for k in results.keys()})

pval_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
pval_fdr_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
effect_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)

for (prs, diag), res in results.items():
    pval_df.loc[diag, prs] = res["p_value"]
    pval_fdr_df.loc[diag, prs] = res["p_value_fdr"]
    effect_df.loc[diag, prs] = res["observed"]

stats_df_wide = pval_fdr_df.copy().clip(lower=EPSILON_VAL)

# --- Process 3: Scale Transforms for Dot Dimensions ---
df_log_transformed = -np.log10(stats_df_wide)
min_val = stats_df_wide.min().min()
max_val = stats_df_wide.max().max()

max_val_log = df_log_transformed.min().min()
min_val_log = df_log_transformed.max().max()

normalized_df_log = (df_log_transformed - min_val_log) / (max_val_log - min_val_log)
dot_size_df_log = MIN_DOT_SIZE + (MAX_DOT_SIZE - MIN_DOT_SIZE) * (1 - normalized_df_log)

# Create binary status matrices (1 for significant, 0 otherwise)
binary_stats_df = stats_df_wide.applymap(lambda x: 1 if x < 0.05 else 0)
star_size = binary_stats_df * dot_size_df_log * 0.7

# --- Process 4: Calculate Contrast differences vs Control ---
median_df = permute_df.groupby(DIAGNOSIS_COL)[prs_list].median()
con_values = median_df.loc['CON']
median_df_con_difference = median_df.subtract(con_values, axis=1).drop('CON')

# Align dimensions to the correct visual category layout
plot_diag_order = [diag for diag in DIAGNOSIS_SELECTOR if diag != "CON"]

dot_size_df_log = dot_size_df_log.reindex(plot_diag_order)[PRS_INTEREST]
median_df_con_difference = median_df_con_difference.reindex(plot_diag_order)[PRS_INTEREST]
binary_stats_df = binary_stats_df.reindex(plot_diag_order)[PRS_INTEREST]
star_size = star_size.reindex(plot_diag_order)[PRS_INTEREST]

# --- Process 5: Load External GWAS Cohort Data ---
heritability_df = pd.read_csv(CASE_DATA_PATH)
heritability_bar_df = heritability_df.set_index("Disorder")

if "Neurodegenerative" not in heritability_bar_df.index:
    heritability_bar_df.loc["Neurodegenerative"] = 0

heritability_bar_df = heritability_bar_df.reindex(PRS_INTEREST)
heritability_bar_df['value'] = heritability_bar_df['Cases'] * -0.00001
heritability_bar = heritability_bar_df['value'].tolist()
positions = np.arange(start=.5, stop=.5 * len(heritability_bar) * 2, step=1)

# Generate cohort percentage distribution bars
diagnosis_counts = permute_df[DIAGNOSIS_COL].value_counts()
diagnosis_bar_df = pd.DataFrame(diagnosis_counts).reset_index()
diagnosis_bar_df.columns = ['Diagnosis', 'Number_of_Observations']
diagnosis_bar_df.set_index('Diagnosis', inplace=True).drop(index='CON', errors='ignore')

total_observations = diagnosis_bar_df['Number_of_Observations'].sum()
diagnosis_bar_df['prop'] = (diagnosis_bar_df['Number_of_Observations'] / total_observations) * -4

diagnosis_bar_df = diagnosis_bar_df.reindex(plot_diag_order)
diagnosis_bar = diagnosis_bar_df['prop'].tolist()
diag_pos = np.arange(start=.5, stop=.5 * (len(diagnosis_bar)) * 2, step=1)



In [ ]:

# ==============================================================================
# 5. OUTPUT: Visualizations & Saving File
# ==============================================================================

sns.set_theme(
    style="darkgrid", 
    font_scale=1.5, 
    rc={'axes.facecolor': '#F0E6EB', "grid.linestyle": "-", "grid.color": '#b0b0b0'}
)

fig = plt.figure(figsize=(20, 8), dpi=600)

# Configure layout gridspec positions
ax1 = plt.subplot2grid((9, 8), (0, 0), colspan=4, rowspan=9) 
ax2 = plt.subplot2grid((9, 8), (0, 4), colspan=1, rowspan=1) 
ax3 = plt.subplot2grid((9, 8), (3, 4), colspan=1, rowspan=1) 

# Scatter heatmap using 'scattermap'
ax1 = scattermap(
    median_df_con_difference,
    cmap=sns.color_palette("Spectral_r", as_cmap=True),
    marker_size=dot_size_df_log,
    ax=ax1,
    vmin=-0.5,
    vmax=0.5,
    linecolor='black',
    linewidths=0.2,
    cbar=True,
    cbar_kws={"shrink": .5}
)

# Overlay categorical asterisks for significance
star_cmap = mcolors.ListedColormap([STAR_COLOR])
ax1 = scattermap(
    binary_stats_df,
    marker='*',
    marker_size=star_size,
    cbar=False,
    ax=ax1,
    linecolor='black',
    linewidths=0.2,
    cmap=star_cmap
)

# Align visual axes
ax1.xaxis.tick_top() 
ax1.tick_params('x', labelrotation=90)
ax1.tick_params('y', labelrotation=0)

cbar = ax1.collections[0].colorbar
cbar.ax.text(
    4, 1.2, 
    'diff in Zscore norm \nPRS case vs control', 
    ha='center', va='center', fontsize=12, transform=cbar.ax.transAxes
)

# Render background layout bar charts
ax1.barh(diag_pos, diagnosis_bar, 0.6, alpha=1, edgecolor="none") 
ax1.axvline(x=0, color='k')
ax1.axhline(0, color='k')

ax1.set_xlim([min(diagnosis_bar) - 0.1, len(prs_list)])
ax1.bar(list(positions)[:11], heritability_bar[:11], 0.6, color='#41b6c4', bottom=0, edgecolor="none")
ax1.set_ylim([len(plot_diag_order), heritability_bar_df['value'].min() - 0.1])

ax1.set_ylabel('')
ax1.set_xlabel('')

# --- Legending Tricks for Dot Sizes ---
x, y = [2, 5], [2, 5]
a2 = [MIN_DOT_SIZE, MAX_DOT_SIZE]
sc = ax2.scatter(x, y, s=a2, alpha=0.5, c='white')
L = ax2.legend(*sc.legend_elements("sizes"), loc='center right', bbox_to_anchor=(1, 0.7), frameon=False)
L.set_title("FDR corrected p-value", prop={'size': 12})
L.get_texts()[1].set_text(round(min_val, 2))
L.get_texts()[0].set_text(round(max_val, 2))
ax2.axis('off')

# --- Legending Tricks for Significance Stars ---
x2, y2 = [4], [2]
a3 = [MAX_DOT_SIZE]
ax3.scatter(x2, y2, s=a3, alpha=1, c=STAR_COLOR, marker='*')
ax3.legend(['significance'], loc='center right', bbox_to_anchor=(1, 1), frameon=False)
ax3.set_ylim([0, 1])
ax3.set_xlim([0, 1])
ax3.axis('off')

# Keep visual text elements vector-editable
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['font.family'] = 'sans-serif'

# Build output directories if not present
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

plt.savefig(OUTPUT_PATH, bbox_inches='tight')
plt.show()

# Analysis: Sex specific association with PRS (Figure extended 2 c, d)

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import seaborn as sns
from statsmodels.stats.multitest import multipletests

# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- Sex Split Selection ---
# Options: ["M"] for Male-only, ["F"] for Female-only, or ["M", "F"] to run both
SEX_SELECTOR = ["M", "F"] 

# --- Analysis Settings ---
DIAGNOSIS_COL = 'nd_simple'
N_PERMUTATIONS = 10000
RANDOM_SEED = 42

# --- Subset Selections ---
DIAGNOSIS_SELECTOR = ["CON", "AD", 'PD',"DLB","VD","FTD","MND","ATAXIA",#"AD + DLB",
     "PSP", "MS", "MSA", "MDD", "BP", "Other PSYCH"]

PRS_INTEREST = [
    'AD_APOE', 'PD', 'DLB', 'FTD', 'Neurodegenerative', 
    'ALS', 'MS', 'MSA', 'MDD', 'BP', 'SCZ'
]

# --- Visual Styling Configuration ---
MIN_DOT_SIZE = 100
MAX_DOT_SIZE = 300
EPSILON_VAL = 1e-300  
STAR_COLOR = "#E59400"

# --- File Paths ---
CASE_DATA_PATH = "/home/../PRS/gwas_cases.txt"
OUTPUT_DIR = "output"

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. `merged_df`: Master pandas DataFrame containing biological sex ("most_likely_sex"), 
   diagnostics ("nd_simple"), and continuous PRS scores.
2. `scattermap`: Custom plotting package/function loaded in your environment.
3. `gwas_cases.txt` located at CASE_DATA_PATH: A csv file containing 'Disorder' and 'Cases' columns.
"""

# --- Optional Data Loading (Uncomment if starting fresh) ---
# if 'merged_df' not in locals():
#     merged_df = pd.read_csv("output/merged_data.csv")
#     print("Data loaded successfully.")

# Load External GWAS Cohort Data (Only need to load once)
heritability_df = pd.read_csv(CASE_DATA_PATH)


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def permutation_test_prs(
    df,
    prs_col,
    group_label,
    diag_col,
    control_label="CON",
    n_permutations=1000,
    seed=42,
    plot=True
):
    """
    Evaluates distribution differences between a case cohort and control cohort 
    using randomized permutation shuffles.
    """
    control = df.loc[df[diag_col] == control_label, prs_col].dropna().values
    group = df.loc[df[diag_col] == group_label, prs_col].dropna().values

    if len(control) == 0 or len(group) == 0:
        raise ValueError(f"No data for {group_label} vs {control_label} in {prs_col}")

    observed = group.mean() - control.mean()
    all_prs = np.concatenate([control, group])
    n_control = len(control)

    rng = np.random.default_rng(seed)
    perm_diffs = np.zeros(n_permutations)

    for i in range(n_permutations):
        perm = rng.permutation(all_prs)
        perm_control = perm[:n_control]
        perm_group = perm[n_control:]
        perm_diffs[i] = perm_group.mean() - perm_control.mean()

    if plot:
        plt.figure()
        plt.hist(perm_diffs, bins=30, density=True)
        plt.axvline(observed, linestyle="--", linewidth=2, color='red')
        plt.xlabel("Mean PRS difference (group − control)")
        plt.ylabel("Density")
        plt.title(f"{prs_col} PRS | {group_label} vs {control_label}")
        plt.show()

    return {
        "observed": observed,
        "perm_diffs": perm_diffs,
        "p_value": np.mean(np.abs(perm_diffs) >= np.abs(observed))
    }


# ==============================================================================
# 4. PROCESSES & OUTPUT LOOP
# ==============================================================================

# Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

for sex in SEX_SELECTOR:
    print(f"\n======================================================================")
    print(f"RUNNING ANALYSIS FOR SEX: {sex}")
    print(f"======================================================================")
    
    # --- Process 1: Filter Dataset by Diagnosis AND Biological Sex ---
    sex_filtered_df = merged_df[
        (merged_df[DIAGNOSIS_COL].isin(DIAGNOSIS_SELECTOR)) & 
        (merged_df["most_likely_sex"] == sex)
    ].copy()
    
    print(f"Total observations for sex '{sex}': {len(sex_filtered_df)}")
    print(sex_filtered_df[DIAGNOSIS_COL].value_counts())
    
    if len(sex_filtered_df) == 0:
        print(f"Skipping sex '{sex}' due to empty dataframe.")
        continue

    # --- Process 2: Pairwise Loop Permutation Testing ---
    results = {}
    for prs in PRS_INTEREST:
        for group in DIAGNOSIS_SELECTOR:
            try:
                res = permutation_test_prs(
                    df=sex_filtered_df,
                    prs_col=prs,
                    group_label=group,
                    diag_col=DIAGNOSIS_COL,
                    control_label="CON",
                    n_permutations=N_PERMUTATIONS,
                    seed=RANDOM_SEED,
                    plot=False
                )
                results[(prs, group)] = res
            except ValueError:
                continue

    if not results:
        print(f"No valid permutation tests computed for sex '{sex}'. Skipping visual plots.")
        continue

    # --- Process 3: Multiple Testing FDR Corrections ---
    keys = list(results.keys())
    pvals = np.array([results[k]["p_value"] for k in keys])

    # Apply Benjamini–Hochberg False Discovery Rate correction
    _, pvals_fdr, _, _ = multipletests(pvals, method="fdr_bh")

    for k, p_fdr in zip(keys, pvals_fdr):
        results[k]["p_value_fdr"] = p_fdr

    # Convert summary stats back to wide formats
    prs_list = sorted({k[0] for k in results.keys()})
    diag_list = sorted({k[1] for k in results.keys()})

    pval_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
    pval_fdr_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)
    effect_df = pd.DataFrame(index=diag_list, columns=prs_list, dtype=float)

    for (prs, diag), res in results.items():
        pval_df.loc[diag, prs] = res["p_value"]
        pval_fdr_df.loc[diag, prs] = res["p_value_fdr"]
        effect_df.loc[diag, prs] = res["observed"]

    stats_df_wide = pval_fdr_df.copy().clip(lower=EPSILON_VAL)

    # --- Process 4: Scale Transformations for Dot Dimensions ---
    df_log_transformed = -np.log10(stats_df_wide)
    min_val = stats_df_wide.min().min()
    max_val = stats_df_wide.max().max()

    max_val_log = df_log_transformed.min().min()
    min_val_log = df_log_transformed.max().max()

    # Avoid zero division when transforming log parameters
    if max_val_log != min_val_log:
        normalized_df_log = (df_log_transformed - min_val_log) / (max_val_log - min_val_log)
    else:
        normalized_df_log = df_log_transformed * 0.0

    dot_size_df_log = MIN_DOT_SIZE + (MAX_DOT_SIZE - MIN_DOT_SIZE) * (1 - normalized_df_log)

    # Create binary status matrices (1 for significant, 0 otherwise)
    binary_stats_df = stats_df_wide.applymap(lambda x: 1 if x < 0.05 else 0)
    star_size = binary_stats_df * dot_size_df_log * 0.7

    # --- Process 5: Calculate Contrast differences vs Control ---
    median_df = sex_filtered_df.groupby(DIAGNOSIS_COL)[prs_list].median()
    
    if 'CON' in median_df.index:
        con_values = median_df.loc['CON']
        median_df_con_difference = median_df.subtract(con_values, axis=1).drop('CON')
    else:
         # Fallback to zeros if control group has zero records inside this specific sex slice
         print(f"Warning: Control group 'CON' is missing for sex: {sex}. Zero baseline differences used.")
         median_df_con_difference = median_df.copy()

    # Align dimensions to the correct visual category layout
    plot_diag_order = [diag for diag in DIAGNOSIS_SELECTOR if diag != "CON"]

    dot_size_df_log = dot_size_df_log.reindex(plot_diag_order, fill_value=MIN_DOT_SIZE)[PRS_INTEREST]
    median_df_con_difference = median_df_con_difference.reindex(plot_diag_order, fill_value=0.0)[PRS_INTEREST]
    binary_stats_df = binary_stats_df.reindex(plot_diag_order, fill_value=0)[PRS_INTEREST]
    star_size = star_size.reindex(plot_diag_order, fill_value=0.0)[PRS_INTEREST]

    # --- Process 6: Set Up Background GWAS Metrics ---
    heritability_bar_df = heritability_df.set_index("Disorder")
    if "Neurodegenerative" not in heritability_bar_df.index:
        heritability_bar_df.loc["Neurodegenerative"] = 0

    heritability_bar_df = heritability_bar_df.reindex(PRS_INTEREST)
    heritability_bar_df['value'] = heritability_bar_df['Cases'] * -0.00001
    heritability_bar = heritability_bar_df['value'].tolist()
    positions = np.arange(start=.5, stop=.5 * len(heritability_bar) * 2, step=1)

    # Generate cohort percentage distribution bars
    diagnosis_counts = sex_filtered_df[DIAGNOSIS_COL].value_counts()
    diagnosis_bar_df = pd.DataFrame(diagnosis_counts).reset_index()
    diagnosis_bar_df.columns = ['Diagnosis', 'Number_of_Observations']
    diagnosis_bar_df.set_index('Diagnosis', inplace=True).drop(index='CON', errors='ignore')

    total_observations = diagnosis_bar_df['Number_of_Observations'].sum()
    if total_observations > 0:
        diagnosis_bar_df['prop'] = (diagnosis_bar_df['Number_of_Observations'] / total_observations) * -4
    else:
        diagnosis_bar_df['prop'] = 0.0

    diagnosis_bar_df = diagnosis_bar_df.reindex(plot_diag_order, fill_value=0.0)
    diagnosis_bar = diagnosis_bar_df['prop'].tolist()
    diag_pos = np.arange(start=.5, stop=.5 * (len(diagnosis_bar)) * 2, step=1)

    # ==========================================================================
    # 5. OUTPUT: Visualizations & Saving File
    # ==========================================================================

    sns.set_theme(
        style="darkgrid", 
        font_scale=1.5, 
        rc={'axes.facecolor': '#F0E6EB', "grid.linestyle": "-", "grid.color": '#b0b0b0'}
    )

    fig = plt.figure(figsize=(20, 8), dpi=600)

    # Grid layout coordinates
    ax1 = plt.subplot2grid((9, 8), (0, 0), colspan=4, rowspan=9) 
    ax2 = plt.subplot2grid((9, 8), (0, 4), colspan=1, rowspan=1) 
    ax3 = plt.subplot2grid((9, 8), (3, 4), colspan=1, rowspan=1) 

    # Scatter heatmap using 'scattermap'
    ax1 = scattermap(
        median_df_con_difference,
        cmap=sns.color_palette("Spectral_r", as_cmap=True),
        marker_size=dot_size_df_log,
        ax=ax1,
        vmin=-0.5,
        vmax=0.5,
        linecolor='black',
        linewidths=0.2,
        cbar=True,
        cbar_kws={"shrink": .5}
    )

    # Overlay categorical asterisks for significance
    star_cmap = mcolors.ListedColormap([STAR_COLOR])
    ax1 = scattermap(
        binary_stats_df,
        marker='*',
        marker_size=star_size,
        cbar=False,
        ax=ax1,
        linecolor='black',
        linewidths=0.2,
        cmap=star_cmap
    )

    ax1.xaxis.tick_top() 
    ax1.tick_params('x', labelrotation=90)
    ax1.tick_params('y', labelrotation=0)

    cbar = ax1.collections[0].colorbar
    cbar.ax.text(
        4, 1.2, 
        'diff in Zscore norm \nPRS case vs control', 
        ha='center', va='center', fontsize=12, transform=cbar.ax.transAxes
    )

    # Render background layout bar charts
    ax1.barh(diag_pos, diagnosis_bar, 0.6, alpha=1, edgecolor="none") 
    ax1.axvline(x=0, color='k')
    ax1.axhline(0, color='k')

    ax1.set_xlim([min(diagnosis_bar) - 0.1 if diagnosis_bar else -0.5, len(prs_list)])
    ax1.bar(list(positions)[:11], heritability_bar[:11], 0.6, color='#41b6c4', bottom=0, edgecolor="none")
    ax1.set_ylim([len(plot_diag_order), heritability_bar_df['value'].min() - 0.1])

    ax1.set_ylabel('')
    ax1.set_xlabel('')
    ax1.set_title(f"PRS Distribution Dotmap ({'Male' if sex == 'M' else 'Female'} Cohort)", pad=20)

    # --- Legend: Dot Sizes ---
    x, y = [2, 5], [2, 5]
    a2 = [MIN_DOT_SIZE, MAX_DOT_SIZE]
    sc = ax2.scatter(x, y, s=a2, alpha=0.5, c='white')
    L = ax2.legend(*sc.legend_elements("sizes"), loc='center right', bbox_to_anchor=(1, 0.7), frameon=False)
    L.set_title("FDR corrected p-value", prop={'size': 12})
    L.get_texts()[1].set_text(round(min_val, 2))
    L.get_texts()[0].set_text(round(max_val, 2))
    ax2.axis('off')

    # --- Legend: Significance Stars ---
    x2, y2 = [4], [2]
    a3 = [MAX_DOT_SIZE]
    ax3.scatter(x2, y2, s=a3, alpha=1, c=STAR_COLOR, marker='*')
    ax3.legend(['significance'], loc='center right', bbox_to_anchor=(1, 1), frameon=False)
    ax3.set_ylim([0, 1])
    ax3.set_xlim([0, 1])
    ax3.axis('off')

    # Vector text formats
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype'] = 42
    mpl.rcParams['svg.fonttype'] = 'none' 
    mpl.rcParams['font.family'] = 'sans-serif'

    # Save sex-specific output PDF
    sex_specific_output_name = f"prs_sbayesrc_disorders_sex_{sex}.pdf"
    plt.savefig(os.path.join(OUTPUT_DIR, sex_specific_output_name), bbox_inches='tight')
    plt.show()